In [1]:
# TODO convert off of hyperopt

In [2]:
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
import mlflow
import pyspark.sql.functions as F
import pyspark.sql.functions as f
from pyspark.sql.functions import col
from pyspark.sql import Window
from mlflow.tracking import MlflowClient
from sklearn.ensemble import RandomForestClassifier
from datetime import datetime, timedelta
# from ray import tune
from mlflow.entities import Dataset

/Users/riley.rustad/miniconda3/envs/hls_ml_mimic/lib/python3.12/site-packages/databricks/ml_features/api/proto/feature_catalog_pb2.py:11: UserWarning: google.protobuf.service module is deprecated. RPC implementations should provide code generator plugins which generate code specific to the RPC implementation. service.py will be removed in Jan 2025
  from google.protobuf import service as _service


In [3]:
dbutils.widgets.text('source_schema', 'kp_catalog.mimic_incr')
source_schema = dbutils.widgets.get('source_schema')

dbutils.widgets.text('target_schema', 'kp_catalog.hls_ml')
target_schema = dbutils.widgets.get('target_schema')

dbutils.widgets.text('max_evals', '50')
max_evals = int(dbutils.widgets.get('max_evals'))

dbutils.widgets.text('model_name', 'kp_catalog.hls_ml.hls_ml_demo')
model_name = dbutils.widgets.get('model_name')
model_name = model_name.split('.')[-1]

# Select the number of months of training history that you want to pull
dbutils.widgets.text('training_months_history', '24')
training_months_history = int(dbutils.widgets.get('training_months_history'))

Box(children=(Label(value='source_schema'), Text(value='kp_catalog.mimic_incr')))

Box(children=(Label(value='target_schema'), Text(value='kp_catalog.hls_ml')))

Box(children=(Label(value='max_evals'), Text(value='50')))

Box(children=(Label(value='model_name'), Text(value='kp_catalog.hls_ml.hls_ml_demo')))

Box(children=(Label(value='training_months_history'), Text(value='24')))

In [4]:
retrain_model = dbutils.jobs.taskValues.get(taskKey    = "model_monitor",
                            key        = "retrain_model",
                            default    = True,
                            debugValue = True)
print(retrain_model)
if not retrain_model:
  dbutils.notebook.exit()

True


### Define our Taget Variable `30_DAY_READMISSION`

In [5]:
admissions = spark.table(f'{source_schema}.admissions')

max_adm_date = admissions.select(f.max(f.col('admittime'))).collect()[0][0]
print(max_adm_date)

w = Window.partitionBy("subject_id").orderBy("admittime")

data = (
  admissions
  # We can't definitively say if anyone from the last 30 days has readmitted
  .filter(col('admittime') < f.lit(max_adm_date - timedelta(days=30)))
  # # Limit training data to the last 3 years
  # .filter(col('admittime') > f.lit(max_adm_date - timedelta(days=365*3)))
  # Calculate the target variable
  .withColumn('last_discharge', f.lag(f.col('dischtime')).over(w))
  .withColumn('new_patient', f.when(f.col('last_discharge').isNull(), 1).otherwise(0))
  .withColumn('IS_A_READMISSION', f.when(
      f.col('last_discharge') > f.date_trunc('dd', f.col('admittime')) - f.expr('INTERVAL 30 DAYS'), 1
  ).otherwise(0))
  .withColumn('30_DAY_READMISSION', f.coalesce(f.lead('IS_A_READMISSION').over(w), f.lit(0)))
  .select('hadm_id', 'subject_id', 'admittime', 'dischtime', '30_DAY_READMISSION')
  .orderBy(['admittime'], desc=True)
)
  
# data.select(f.max(f.col('admittime'))).collect()[0][0]

2025-02-12 16:00:00


### Train/Test Split

In [6]:
training_data = (
  data
  # We can't definitively say if anyone from the last 30 days has readmitted
  .filter(col('admittime') < f.lit(max_adm_date - timedelta(days=90)))
  .filter(col('admittime') > f.lit(max_adm_date - timedelta(days=training_months_history*30) - timedelta(days=90)))
  .drop('admittime')
)

validation_data = (
  data
  # We can't definitively say if anyone from the last 30 days has readmitted
  .filter(col('admittime') < f.lit(max_adm_date - timedelta(days=60)))
  .filter(col('admittime') > f.lit(max_adm_date - timedelta(days=90)))
  .drop('admittime')
)

### Define What Features To Look Up

In [7]:
display(spark.table(f"{source_schema}.patients").select('gender').distinct())

,gender
0,F
1,M


In [8]:
patient_feature_lookups = [
   FeatureLookup( 
     table_name = f"{target_schema}.patient_features",
     feature_names = [
      'gender_f',
      'gender_m'],
     lookup_key = ["subject_id"],
   ),
]
 
admissions_feature_lookups = [
   FeatureLookup( 
     table_name = f"{target_schema}.admissions_features",
     feature_names = [
        "admission_type_direct_observation",
        "admission_type_eu_observation",
        "admission_type_ew_emer",
        "admission_type_elective",
        "admission_type_surgical_same_day_admission",
        "admission_type_observation_admit",
        "admission_type_ambulatory_observation",
        "admission_type_direct_emer",
        "admission_type_urgent",
        "admission_location_internal_transfer_to_or_from_psych",
        "admission_location_procedure_site",
        "admission_location_emergency_room",
        "admission_location_physician_referral",
        "admission_location_transfer_from_skilled_nursing_facility",
        "admission_location_walk_in_self_referral",
        "admission_location_clinic_referral",
        "admission_location_pacu",
        "admission_location_transfer_from_hospital",
        "admission_location_information_not_available",
        "admission_location_ambulatory_surgery_transfer",
        "insurance_private",
        "insurance_other",
        "insurance_medicaid",
        "insurance_no_charge",
        "insurance_medicare",
        "insurance_none",
        "marital_status_widowed",
        "marital_status_single",
        "marital_status_married",
        "marital_status_divorced",
        "marital_status_none",
     ],
     lookup_key = ["hadm_id"],
   ),
]

age_at_enc_feature_lookups = [
   FeatureLookup( 
     table_name = f"{target_schema}.age_at_admission",
     feature_names = ['age_at_admission'],
     lookup_key = ["hadm_id"],
   ),
]

historic_admission_feature_lookups = [
   FeatureLookup( 
     table_name = f"{target_schema}.historic_admissions_features",
     feature_names = [
        "new_patient",
        "IS_A_READMISSION",
        "30_DAY_READMISSION_6_months",
        "30_DAY_READMISSION_12_months",
        "prev_admissions_6_months",
        "prev_admissions_12_months",
     ],
     lookup_key = ["hadm_id"],
   ),
]

### Use Feature Store to Create Dataset Based on Lookups

In [9]:

fe = FeatureEngineeringClient()
training_set = fe.create_training_set(
  df = data,
  feature_lookups = patient_feature_lookups + admissions_feature_lookups + age_at_enc_feature_lookups + historic_admission_feature_lookups,
  label = "30_DAY_READMISSION",
  exclude_columns = ["hadm_id", "subject_id", 'dischtime', 'admittime']
)

enriched = training_set.load_df()

train = (
  enriched
  # We can't definitively say if anyone from the last 30 days has readmitted
  .filter(col('admittime') < f.lit(max_adm_date - timedelta(days=90)))
  .filter(col('admittime') > f.lit(max_adm_date - timedelta(days=training_months_history*30) - timedelta(days=90)))
  .drop('admittime')
).toPandas()

val = (
  enriched
  # We can't definitively say if anyone from the last 30 days has readmitted
  .filter(col('admittime') < f.lit(max_adm_date - timedelta(days=60)))
  .filter(col('admittime') > f.lit(max_adm_date - timedelta(days=90)))
  .drop('admittime')
).toPandas()



In [10]:
fe = FeatureEngineeringClient()
training_set = fe.create_training_set(
  df = training_data,
  feature_lookups = patient_feature_lookups + admissions_feature_lookups + age_at_enc_feature_lookups + historic_admission_feature_lookups,
  label = "30_DAY_READMISSION",
  exclude_columns = ["hadm_id", "subject_id", 'dischtime', 'admittime']
)

train = training_set.load_df().toPandas()

validation_set = fe.create_training_set(
  df = validation_data,
  feature_lookups = patient_feature_lookups + admissions_feature_lookups + age_at_enc_feature_lookups + historic_admission_feature_lookups,
  label = "30_DAY_READMISSION",
  exclude_columns = ["hadm_id", "subject_id", 'dischtime', 'admittime']
)

val = validation_set.load_df().toPandas()

In [11]:
val.shape,train.shape

((7586, 41), (94479, 41))

In [12]:
train
# train.isnull().values.any()

,gender_f,gender_m,admission_type_direct_observation,admission_type_eu_observation,admission_type_ew_emer,admission_type_elective,admission_type_surgical_same_day_admission,admission_type_observation_admit,admission_type_ambulatory_observation,admission_type_direct_emer,...,marital_status_divorced,marital_status_none,age_at_admission,new_patient,IS_A_READMISSION,30_DAY_READMISSION_6_months,30_DAY_READMISSION_12_months,prev_admissions_6_months,prev_admissions_12_months,30_DAY_READMISSION
0,1,0,0,0,1,0,0,0,0,0,...,0,0,85.776865,1,0.0,0.0,0.0,1,1,0
1,1,0,0,0,1,0,0,0,0,0,...,0,0,85.982204,0,0.0,0.0,0.0,2,2,0
2,1,0,0,0,1,0,0,0,0,0,...,0,0,86.110883,0,0.0,0.0,0.0,3,3,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,37.303217,1,0.0,0.0,0.0,1,1,0
4,1,0,0,0,0,0,0,1,0,0,...,0,0,73.114305,0,0.0,0.0,0.0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94474,0,1,0,0,0,1,0,0,0,0,...,0,0,60.758385,0,1.0,4.0,4.0,5,5,0
94475,0,1,0,0,0,1,0,0,0,0,...,0,0,60.892539,0,0.0,4.0,4.0,6,6,1
94476,0,1,0,0,0,1,0,0,0,0,...,0,0,60.977413,0,1.0,5.0,5.0,7,7,1
94477,0,1,1,0,0,0,0,0,0,0,...,0,0,61.054073,0,1.0,6.0,6.0,8,8,0


### Define the MLFlow Experiement

In [13]:
max_adm_date

datetime.datetime(2025, 2, 12, 16, 0)

In [14]:
from datetime import datetime
demo_date = datetime.today()
demo_date

datetime.datetime(2025, 8, 26, 16, 9, 39, 814709)

In [15]:
#TODO: what is the prod location for an experiment
experiment_name = f"/Users/riley.rustad@databricks.com/{model_name}_{max_adm_date.strftime('%Y%m%d')}"
mlflow.set_experiment(experiment_name)
target_col = "30_DAY_READMISSION"

In [16]:
experiment_name

'/Users/riley.rustad@databricks.com/hls_ml_demo_20250212'

### Iterate Through Hyperparameters
Logging all ML Model parameters, metrics, and artifacts

In [17]:
# TODO: exclude id columns from the training set

In [18]:
import mlflow
import sklearn
# from hyperopt import STATUS_OK, Trials, fmin, hp, tpe, SparkTrials
import mlflow.sklearn
from sklearn.metrics import roc_auc_score

# mlflow.sklearn.autolog(log_models=False)
mlflow.autolog(disable=True)

In [26]:
import mlflow
import mlflow.sklearn
import numpy as np
from ray import tune
from ray.tune.search.optuna import OptunaSearch
from ray.tune.schedulers import ASHAScheduler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

mlflow.autolog(disable=True)  # manual logging for control

def train_fn(config, train, val, target_col):
    with mlflow.start_run():
        # Log params to MLflow
        for key, value in config.items():
            mlflow.log_param(key, value)

        # Train model
        model = RandomForestClassifier(
            n_estimators=int(config['n_estimators']),
            max_depth=int(config['max_depth']),
            min_samples_split=int(config['min_samples_split']),
            random_state=config.get('random_state', 42)
        )

        model.fit(train.drop(columns=target_col), train[target_col])

        # Track datasets (optional, depending on your fe.log_model needs)
        train_dataset = mlflow.data.from_pandas(train, name="train")
        val_dataset = mlflow.data.from_pandas(val, name="val")

        # Log model (swap for your custom function if needed)
        fe.log_model(
            model=model,
            artifact_path="model",
            flavor=mlflow.sklearn,
            training_set=train_dataset,
            infer_input_example=True,
        )

        # Evaluate
        val_preds = model.predict(val.drop(columns=target_col))
        val_auc = roc_auc_score(val[target_col], val_preds)

        train_preds = model.predict(train.drop(columns=target_col))
        train_auc = roc_auc_score(train[target_col], train_preds)

        diff = train_auc - val_auc
        model_info = mlflow.pyfunc.load_model(f"runs:/{mlflow.active_run().info.run_id}/model")

        # Log metrics
        mlflow.log_metric("val_auc", val_auc, model_id=model_info.model_id, dataset=val_dataset)
        mlflow.log_metric("train_auc", train_auc, model_id=model_info.model_id, dataset=train_dataset)
        mlflow.log_metric("diff", diff, model_id=model_info.model_id)

        # Report to Ray Tune
        tune.report(loss=1 - val_auc)

def optimize_with_optuna(train, val, target_col, max_evals=30, random_state=42):
    search_space = {
        "n_estimators": tune.quniform(100, 1000, 1),
        "max_depth": tune.choice(list(range(1, 14))),
        "min_samples_split": tune.quniform(2, 6, 1),
        "random_state": random_state
    }

    # Search algorithm: Optuna (uses TPE by default)
    search_alg = OptunaSearch(metric="loss", mode="min", seed=random_state)

    # Scheduler for early stopping
    scheduler = ASHAScheduler(metric="loss", mode="min")

    # Run the experiment
    tune.run(
        tune.with_parameters(train_fn, train=train, val=val, target_col=target_col),
        config=search_space,
        num_samples=max_evals,
        search_alg=search_alg,
        scheduler=scheduler,
        storage_path="file://./ray_results",
        name="rf_optuna_raytune",
        resources_per_trial={"cpu": 1}
    )



In [30]:
search_space = {
    "n_estimators": tune.quniform(100, 1000, 1),
    "max_depth": tune.choice(list(range(1, 14))),
    "min_samples_split": tune.quniform(2, 6, 1),
    "random_state": 42
}

tuner = tune.Tuner(
    train_fn,
    tune_config=tune.TuneConfig(
        num_samples=max_evals,
        search_alg=search_space,
    ),
)
results = tuner.fit()

2025-08-26 16:22:25,972	INFO tune.py:616 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949


ValueError: Unknown argument found in the Trainable function. The function args must include a single 'config' positional parameter.
Found: ['config', 'train', 'val', 'target_col']

In [ ]:
optimize_with_optuna(train, val, target_col, max_evals)

2025-08-26 16:19:17,460	INFO tune.py:616 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949


ArrowInvalid: Unsupported hostname in non-Windows local URI: 'file://./ray_results'

In [ ]:
# import mlflow
# import sklearn
# from hyperopt import STATUS_OK, Trials, fmin, hp, tpe, SparkTrials
# import mlflow.sklearn
# from sklearn.metrics import roc_auc_score

# # mlflow.sklearn.autolog(log_models=False)
# mlflow.autolog(disable=True)

# def optimize(
#             #  trials, 
#              max_evals,
#              max_evals,
#              random_state=42):
#     """
#     This is the optimization function that given a space (space here) of 
#     hyperparameters and a scoring function (score here), finds the best hyperparameters.
#     """
#     space = {
#         'n_estimators': hp.quniform('n_estimators', 100, 1000, 1),
#         'max_depth':  hp.choice('max_depth', range(1, 14)),
#         'min_samples_split': hp.quniform('min_samples_split', 2, 6, 1),
#         'random_state': random_state
#     }
#     # spark_trials = SparkTrials()
#     # Use the fmin function from Hyperopt to find the best hyperparameters
#     best = fmin(score, space, algo=tpe.suggest, 
#                 # trials=trials, 
#                 max_evals=max_evals)
#     return best
  
# def score(params):
#   with mlflow.start_run() as training_run:

#     train_dataset: Dataset = mlflow.data.from_pandas(train, name="train")
#     validation_dataset: Dataset = mlflow.data.from_pandas(val, name="val")

#     for key, value in params.items():
#       mlflow.log_param(key, value)
    
#     model = RandomForestClassifier(
#       n_estimators = int(params['n_estimators']),
#       max_depth = int(params['max_depth']),
#       min_samples_split = int(params['min_samples_split'])
#     )

#     model.fit(train.drop(target_col,axis=1), train[target_col])

# #     mlflow.sklearn.log_model(model, 'model')
#     fe.log_model(
#       model=model,
#       artifact_path="model",
#       flavor=mlflow.sklearn,
#       training_set=training_set,
#       infer_input_example=True,
#       # dataset=train_dataset
#       # registered_model_name="kp_catalog.hls_ml.readmissions"
#     )

#     preds = model.predict(val.drop(target_col,axis=1))
#     score = roc_auc_score(val[target_col], preds)
    
#     train_preds = model.predict(train.drop(target_col,axis=1))
#     train_score = roc_auc_score(train[target_col], train_preds)

#   #   mlflow.log_metrics(
#   #     metrics={
#   #       "rmse": rmse,
#   #       "r2": r2,
#   #       "mae": mae,
#   #     }, 
#   #   dataset=test_dataset,
#   #   model_id=logged_model.model_id
#   # )

#     model_info = mlflow.pyfunc.load_model(f"runs:/{mlflow.active_run().info.run_id}/model")

#     mlflow.log_metric('val_auc', score, model_id=model_info.model_id, dataset=validation_dataset)
#     mlflow.log_metric('train_auc', train_score, model_id=model_info.model_id, dataset=train_dataset)
#     # I like to compare val and train metrics so that I can measure overfitting
#     mlflow.log_metric('diff', train_score - score,model_id=model_info.model_id)
  
#     loss = 1 - score
#   return {'loss': loss, 'status': STATUS_OK}


In [0]:
# # spark_trials = SparkTrials()
# best_hyperparams = optimize(
#                             # trials = spark_trials,
#                             max_evals
#                             )

In [0]:
# dbutils.jobs.taskValues.set(key= "experiment_name",value = experiment_name)